<a href="https://colab.research.google.com/github/AbhinavPal108/EMOTE-Emotion-Detection/blob/main/Emote.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import re

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Bidirectional, Dense, Dropout
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [ ]:
from google.colab import files
files.upload()


In [ ]:
df = pd.read_csv("/content/goemotions_1.csv")

print("Columns found:", df.columns.tolist())

# Case 1: Dataset already has label column
if 'label' in df.columns:
    df = df[['text', 'label']].copy()
    print("Using existing text + label dataset")

# Case 2: Dataset has one-hot emotion columns
else:
    # Added link_id, parent_id, created_utc, rater_id to remove_cols as they are metadata
    remove_cols = ['id', 'example_very_unclear', 'author', 'subreddit', 'link_id', 'parent_id', 'created_utc', 'rater_id']
    emotion_cols = [col for col in df.columns if col not in ['text'] + remove_cols]

    if len(emotion_cols) == 0:
        raise ValueError("No emotion columns found in dataset")

    # keep rows having at least one emotion
    df = df[df[emotion_cols].sum(axis=1) > 0].copy()

    # convert one-hot to single label
    df['label'] = df[emotion_cols].idxmax(axis=1)

    df = df[['text', 'label']].copy()
    print("Converted one-hot emotions into labels")

print(df.head())
print("Final shape:", df.shape)

In [ ]:
def clean_text(text):
    return re.sub(r'[^a-zA-Z\s]', '', text.lower())

df['text'] = df['text'].apply(clean_text)


In [ ]:
le = LabelEncoder()
df['label_encoded'] = le.fit_transform(df['label'])


In [ ]:
# remove null texts
df = df.dropna(subset=['text'])

# convert everything safely to string
df['text'] = df['text'].astype(str)

# remove blank rows
df = df[df['text'].str.strip() != '']

print(df['text'].isnull().sum())
print(df.shape)

In [ ]:
tokenizer = Tokenizer(
    num_words=80000,
    oov_token="<OOV>"
)

# VERY IMPORTANT
tokenizer.fit_on_texts(df['text'].astype(str).tolist())

MAX_LEN = 200

sequences = tokenizer.texts_to_sequences(
    df['text'].astype(str).tolist()
)

X = pad_sequences(
    sequences,
    maxlen=MAX_LEN,
    padding='post',
    truncating='post',
    value=0
)

y = df['label_encoded'].values

In [ ]:
print(df.columns)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [ ]:
model = Sequential([
    Embedding(80000, 128, input_length=MAX_LEN),
    Bidirectional(LSTM(128, return_sequences=True)),
    Dropout(0.5),

    Bidirectional(LSTM(64)),
    Dropout(0.4),

    Dense(256, activation='relu'),
    Dropout(0.3),

    Dense(len(le.classes_), activation='softmax')
])

In [ ]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)


In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weights = dict(enumerate(class_weights))


In [ ]:
model.fit(
    X_train, y_train,
    epochs=25,
    batch_size=32,
    validation_data=(X_test, y_test),
    class_weight=class_weights
)


In [ ]:
y_pred = np.argmax(model.predict(X_test), axis=1)
print(classification_report(y_test, y_pred, target_names=le.classes_))


In [ ]:
MAX_LEN = 150

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    return text.strip()

def predict_emotion(text):
    text = clean_text(text)
    seq = tokenizer.texts_to_sequences([text])
    pad = pad_sequences(seq, maxlen=MAX_LEN, padding='post', truncating='post')
    pred = np.argmax(model.predict(pad, verbose=0), axis=1)[0]
    return le.inverse_transform([pred])[0]

# 10 demo statements
tests = [
    "I feel very tired and broken today",
    "I feel amazing after getting good marks",
    "I feel extremely happy after winning the competition",
    "I am deeply hurt and broken today",
    "I am so angry at what happened",
    "I am excited for tomorrow's trip",
    "I feel scared about the exam results",
    "Thank you so much for helping me",
    "I miss my old friends and feel lonely",
    "I love spending time with my family"
]

for t in tests:
    print(f"{t} --> {predict_emotion(t)}")